In [ ]:
import pandas as pd
import re
import json
from transformers import pipeline

In [29]:
data = pd.read_excel('prague_emotional_data.xlsx')

In [30]:
print(f"Total number of samples: {len(data)}")
print(f"Number of null samples: {data['comment_EN'].isnull().sum()}")

# Filter data for usable comments
filtered_data = data[data['comment_EN'].notnull()].copy()
print(f"Number of usable samples: {len(filtered_data)}")

Total number of samples: 733
Number of null samples: 0
Number of usable samples: 733


In [31]:
# Clean text, lowercase, remove extra spaces, kept punctuation
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

filtered_data['clean_comment'] = filtered_data['comment_EN'].apply(clean_text)

# reset_index() if you want to have indexes from 0 to len(filtered_data). convenient for batching
filtered_data.reset_index(inplace=True)

# Hugging Face

In [32]:
#sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

Device set to use cpu


In [33]:
### EXAMPLE: Test sentiment analysis on the first 5 comments ###

for comment in filtered_data['clean_comment'].head(5).tolist():
    result = sentiment_pipeline(comment)
    print(f"Input: {comment}")
    print(f"Output: {result}\n")

Input: they tend to be crowded
Output: [{'label': 'NEGATIVE', 'score': 0.9952652454376221}]

Input: a fundamental lack of parking
Output: [{'label': 'NEGATIVE', 'score': 0.999693751335144}]

Input: wild parking even in front of crossings - dangerous for pedestrians and cars
Output: [{'label': 'NEGATIVE', 'score': 0.9711722135543823}]

Input: overcrowded dog poop bins around lannova park.
Output: [{'label': 'NEGATIVE', 'score': 0.9982789754867554}]

Input: the parked cars under the charles bridge spoil the place terribly.
Output: [{'label': 'NEGATIVE', 'score': 0.9991254210472107}]



In [34]:
def hf_batch_inference(df, batch_size=10, output_name='prague_sentiment_results_hf.csv'):

    # Batch processing for sentiment alaysis
    count = 0
    results_df = pd.DataFrame()

    for start in range(0, len(df), batch_size):
        end = start + batch_size
        batch_df = df.iloc[start:end].copy()
        batch_comments = batch_df['clean_comment'].tolist()
        batch_results = sentiment_pipeline(batch_comments)
        count += len(batch_results)
        
        batch_df['sentiment_label'] = [r['label'] for r in batch_results]
        batch_df['sentiment_score'] = [r['score'] for r in batch_results]
        
        results_df = pd.concat([results_df, batch_df], ignore_index=True)
        results_df.drop(columns="index", inplace=True)
        results_df.to_csv(output_name, index=False)
        
        print(f"Processed {count}/{len(df)} comments")  # Progress update

In [35]:
hf_batch_inference(filtered_data, output_name="prague_sentiment_results.csv")

Processed 10/733 comments
Processed 20/733 comments
Processed 30/733 comments
Processed 40/733 comments
Processed 50/733 comments
Processed 60/733 comments
Processed 70/733 comments
Processed 80/733 comments
Processed 90/733 comments
Processed 100/733 comments
Processed 110/733 comments
Processed 120/733 comments
Processed 130/733 comments
Processed 140/733 comments
Processed 150/733 comments
Processed 160/733 comments
Processed 170/733 comments
Processed 180/733 comments
Processed 190/733 comments
Processed 200/733 comments
Processed 210/733 comments
Processed 220/733 comments
Processed 230/733 comments
Processed 240/733 comments
Processed 250/733 comments
Processed 260/733 comments
Processed 270/733 comments
Processed 280/733 comments
Processed 290/733 comments
Processed 300/733 comments
Processed 310/733 comments
Processed 320/733 comments
Processed 330/733 comments
Processed 340/733 comments
Processed 350/733 comments
Processed 360/733 comments
Processed 370/733 comments
Processed 